# 00 - Match selection, AWS access, and local cache

Select seasons, tracked sources, and optional match dates here. Local NFF schedule exports are required only for an AWS rebuild and are not distributed in the public repository. The final build cell runs only when `REBUILD_CACHE_FROM_AWS = True`.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


# Locate the repository before importing its analysis package. This works when
# Jupyter starts from either the repo root or this notebook directory.
_start = Path.cwd()
ROOT = next(
    path for path in (_start, *_start.parents)
    if (path / "analysis" / "levy_paper").is_dir()
    and (path / "requirements.txt").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from analysis.levy_paper.util.publication_notebook_utils import (
    PRIMARY_CACHE_SUFFIX,
    cache_path as make_cache_path,
    csv_shapes,
    display_live_or_frozen,
    file_status,
    hazard_support_summary,
    load_processed_cache,
    order_state_summary,
    panel_inventory,
    publication_paths,
    relative_path,
    resolve_data_mode,
    table_inventory,
    transition_row_sum_audit,
    transport_run_summary,
)

PATHS = publication_paths(ROOT)
LEVY_DIR = PATHS["levy_dir"]
DATA_DIR = PATHS["data_dir"]
PRIMARY_CACHE_DIR = PATHS["primary_cache_dir"]
FINAL_FIGURES = PATHS["final_figures"]
SOURCE_DATA = PATHS["source_data"]
SUPPLEMENT = PATHS["supplement"]
CACHE_SUFFIX = PRIMARY_CACHE_SUFFIX

# DATA_MODE options:
#   "auto"     use processed caches when all required files exist;
#              otherwise use tracked reviewer tables/frozen figures
#   "cache"    require processed caches and fail clearly if they are missing
#   "reviewer" use only tracked public artefacts
DATA_MODE = "auto"
BUILD_FIGURE = True
SAVE_FIGURE_OUTPUTS = True
DISPLAY_FROZEN_OUTPUT = True
REBUILD_CACHE_FROM_AWS = False
REFIT_FIGURE4_FIGURE5_MODELS = False
USE_VERSIONED_FINAL_FIGURE4_FIT = True


def rel(path):
    return relative_path(path, ROOT)


def show_file_status(paths):
    return file_status(paths, ROOT)


def show_csv_shapes(paths):
    return csv_shapes(paths, ROOT)


def cache_path(stem):
    return make_cache_path(PRIMARY_CACHE_DIR, stem, CACHE_SUFFIX)


def show_figure(fig, frozen_path, width=1100):
    return display_live_or_frozen(
        fig,
        frozen_path,
        display_frozen=DISPLAY_FROZEN_OUTPUT,
        width=width,
    )

## Analysis selection

In [ ]:
SELECTED_SEASONS = [2020, 2021]
SELECTED_SOURCES = ["A", "B"]
SELECTED_MATCH_DATES = None  # Example: ["2021-05-22", "2021-06-30"]
INCLUDE_SINGLE_TEAM_FILES = True
INCLUDE_HEAD_TO_HEAD_FILES = True
REQUIRE_MATCH_WINDOW = True

# Use a new name/suffix for exploratory subsets. The canonical names below
# reproduce the paper cache and should be overwritten only deliberately.
OUTPUT_CACHE_NAME = "all_team_2020_2021_sticky_active"
OUTPUT_SUFFIX = "2020_2021_all_teams_pitchfix_sticky_active"
OVERWRITE_EXISTING_CACHE = False
MAX_SINGLE_MATCHES = None
MAX_H2H_MATCHES = None
MAX_FILES_PER_MATCH = None

# Local load controls. The frame-level trajectory and full player-run tables
# are opt-in because they expand substantially in memory after parquet decoding.
LOAD_ANALYSIS_TABLES = True
LOAD_CENTROID_RUNS = True
LOAD_FULL_RUN_TABLE = False
LOAD_TRAJECTORY_TABLE = False
TRAJECTORY_COLUMNS = [
    "match_id", "match_phase", "team", "source_key", "track_type",
    "track_entity_uid", "player_name", "t", "x_m", "y_m", "season",
]

## Schedule selection

In [ ]:
schedule_paths = {
    2020: LEVY_DIR / "metadata" / "schedules" / "kamper_2020.xlsx",
    2021: LEVY_DIR / "metadata" / "schedules" / "kamper_2021.xlsx",
}
schedule_frames = []
missing_schedule_paths = [
    schedule_paths[season]
    for season in SELECTED_SEASONS
    if not schedule_paths[season].exists()
]
if missing_schedule_paths:
    schedule = pd.DataFrame()
    print("local_schedule_inputs_missing")
    for path in missing_schedule_paths:
        print(" -", rel(path))
    print("required_only_for_aws_rebuild", rel(LEVY_DIR / "metadata" / "schedules" / "README.md"))
else:
    for season in SELECTED_SEASONS:
        path = schedule_paths[season]
        frame = pd.read_excel(path)
        frame["season"] = season
        frame["schedule_source"] = rel(path)
        schedule_frames.append(frame)
    schedule = pd.concat(schedule_frames, ignore_index=True)

if SELECTED_MATCH_DATES and not schedule.empty:
    requested_dates = {pd.Timestamp(value).strftime("%Y-%m-%d") for value in SELECTED_MATCH_DATES}
    date_columns = [column for column in schedule.columns if "date" in str(column).lower() or "dato" in str(column).lower()]
    if not date_columns:
        raise KeyError("No date-like schedule column found.")
    schedule_dates = pd.to_datetime(schedule[date_columns[0]], errors="coerce").dt.strftime("%Y-%m-%d")
    schedule = schedule.loc[schedule_dates.isin(requested_dates)].copy()

print("selected_schedule_rows", len(schedule))
display(schedule.head(20))

## Pitch registry and processed cache

In [ ]:
pitch_registry = LEVY_DIR / "metadata" / "pitches" / "toppserien_pitches.json"
primary_cache_files = [
    cache_path("trajectory_long"),
    cache_path("runs_long"),
    cache_path("msd_long"),
    cache_path("df_pmv"),
    cache_path("centroid_order_runs"),
    cache_path("hazard_intervals"),
]
display(show_file_status([pitch_registry, *schedule_paths.values(), *primary_cache_files]))

cache_inventory = []
for path in primary_cache_files:
    row = {"file": rel(path), "exists": path.exists()}
    if path.exists():
        sample = pd.read_parquet(path).head(3)
        row.update({"columns": len(sample.columns), "sample_rows": len(sample)})
    cache_inventory.append(row)
display(pd.DataFrame(cache_inventory))

## Build command

In [ ]:
builder = LEVY_DIR / "scripts" / "build_multiseason_data_cache.py"
build_command = [
    sys.executable, str(builder),
    "--seasons", *map(str, SELECTED_SEASONS),
    "--single-sources", *SELECTED_SOURCES,
    "--cache-name", OUTPUT_CACHE_NAME,
    "--suffix", OUTPUT_SUFFIX,
]
if not INCLUDE_SINGLE_TEAM_FILES:
    build_command.append("--skip-single")
if not INCLUDE_HEAD_TO_HEAD_FILES:
    build_command.append("--skip-h2h")
if not REQUIRE_MATCH_WINDOW:
    build_command.append("--no-require-match-window")
if SELECTED_MATCH_DATES:
    build_command.extend(["--match-dates", *SELECTED_MATCH_DATES])
if MAX_SINGLE_MATCHES is not None:
    build_command.extend(["--max-single-matches", str(MAX_SINGLE_MATCHES)])
if MAX_H2H_MATCHES is not None:
    build_command.extend(["--max-h2h-matches", str(MAX_H2H_MATCHES)])
if MAX_FILES_PER_MATCH is not None:
    build_command.extend(["--max-files", str(MAX_FILES_PER_MATCH)])
if OVERWRITE_EXISTING_CACHE:
    build_command.append("--overwrite")

print(" ".join(build_command))

## Optional AWS cache build

In [ ]:
canonical_subset_risk = bool(SELECTED_MATCH_DATES) and (
    OUTPUT_CACHE_NAME == "all_team_2020_2021_sticky_active"
    or OUTPUT_SUFFIX == "2020_2021_all_teams_pitchfix_sticky_active"
)
if REBUILD_CACHE_FROM_AWS:
    if missing_schedule_paths:
        missing = "\n".join(f" - {rel(path)}" for path in missing_schedule_paths)
        raise FileNotFoundError(
            "AWS rebuild requires local NFF schedule inputs:\n"
            f"{missing}\n"
            f"See {rel(LEVY_DIR / 'metadata' / 'schedules' / 'README.md')}"
        )
    if canonical_subset_risk:
        raise ValueError("Choose a new OUTPUT_CACHE_NAME and OUTPUT_SUFFIX for a date subset.")
    subprocess.run(build_command, cwd=ROOT, check=True)
else:
    print("AWS build skipped")

## Load processed data into the notebook

In [ ]:
analysis_tables = {}
selected_cache_dir = DATA_DIR / "processed" / OUTPUT_CACHE_NAME
default_analysis_paths = [
    selected_cache_dir / f"{name}_{OUTPUT_SUFFIX}.parquet"
    for name in ["df_pmv", "msd_long", "centroid_order_runs", "hazard_intervals", "runs_long"]
]
resolved_mode = resolve_data_mode(DATA_MODE, default_analysis_paths)
selected_cache_ready = resolved_mode == "cache"
print("resolved_data_mode", resolved_mode)

if LOAD_ANALYSIS_TABLES and selected_cache_ready:
    for table_name in ["df_pmv", "msd_long", "centroid_order_runs", "hazard_intervals"]:
        analysis_tables[table_name] = load_processed_cache(
            selected_cache_dir, table_name, OUTPUT_SUFFIX
        )

    # Named variables make the tables immediately usable in exploratory cells.
    df_pmv = analysis_tables["df_pmv"]
    msd_long = analysis_tables["msd_long"]
    centroid_order_runs = analysis_tables["centroid_order_runs"]
    hazard_intervals = analysis_tables["hazard_intervals"]

centroid_runs = None
if LOAD_CENTROID_RUNS and selected_cache_ready:
    centroid_runs = load_processed_cache(
        selected_cache_dir,
        "runs_long",
        OUTPUT_SUFFIX,
        filters=[("track_type", "==", "centroid")],
    )

runs_long = None
if LOAD_FULL_RUN_TABLE and selected_cache_ready:
    runs_long = load_processed_cache(selected_cache_dir, "runs_long", OUTPUT_SUFFIX)

trajectory_long = None
trajectory_path = selected_cache_dir / f"trajectory_long_{OUTPUT_SUFFIX}.parquet"
if LOAD_TRAJECTORY_TABLE and trajectory_path.exists():
    trajectory_long = load_processed_cache(
        selected_cache_dir,
        "trajectory_long",
        OUTPUT_SUFFIX,
        columns=TRAJECTORY_COLUMNS,
    )

loaded_summary = [
    {"variable": name, "rows": len(frame), "columns": len(frame.columns)}
    for name, frame in {
        **analysis_tables,
        "centroid_runs": centroid_runs,
        "runs_long": runs_long,
        "trajectory_long": trajectory_long,
    }.items()
    if frame is not None
]
display(pd.DataFrame(loaded_summary))
if not selected_cache_ready:
    print("Processed cache unavailable; dataframes were not loaded. Frozen/source-table figure mode remains available.")

## Selected-data examples

In [ ]:
if centroid_runs is not None:
    display(centroid_runs.head())
if analysis_tables:
    display(hazard_intervals.head())
    display(centroid_order_runs.head())